# Silver — physical_itens_venda_caixa

**Regras técnicas aplicadas nesta camada:**
1. `id_item_venda` não pode ser nulo nem duplicado (PK).
2. `id_transacao` deve existir em `physical_vendas_caixa` (FK).
3. `quantidade` deve ser > 0.
4. `preco_unitario_registro` deve ser > 0, `DECIMAL(10,2)`.
5. `valor_total_item` deve ser igual a `preco_unitario_registro * quantidade`.

**Otimização para Serverless:**
Todas as flags de qualidade são calculadas em um único `select()`,
evitando múltiplas varreduras completas de 3,7M linhas.
As métricas de DQ são registradas com um único `count()` final
usando agregações condicionais.

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

# Reduzir shuffle partitions para aliviar o Serverless
# (padrão 200 é excessivo para volumes médios)
spark.conf.set("spark.sql.shuffle.partitions", "32")

SILVER_WRITE_MODE = "overwrite"

print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Modo de escrita   : {SILVER_WRITE_MODE}")

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, when, expr,
    abs as spark_abs, round as spark_round,
    count, sum as spark_sum, udf, floor,
    translate, regexp_replace, lower, upper
)
from pyspark.sql.types import StringType



####Instrução do cliente — Normalização de produto para ranking


codigo_barras_produto hoje é um slug de texto livre (categoria-marca-tamanho-variante), não um SKU/EAN oficial. Pequenas variações de escrita — acentos, espaços, maiúsculas, hífens duplicados — fazem o mesmo produto aparecer como linhas diferentes no ranking de mais vendidos, diluindo sua posição real.

codigo_produto_normalizado resolve isso: minúsculas, sem acento, sem espaço, sem hífen duplicado/nas pontas. Não substitui um SKU oficial (que exigiria um cadastro de produto que não existe hoje), mas agrupa corretamente variações de escrita do mesmo item.

In [0]:
# Performance: em vez de UDF Python (serializa linha a linha entre
# JVM e Python — lento em milhões de linhas, pode até derrubar
# clusters pequenos como o do Databricks Free Edition), a
# normalização usa só funções NATIVAS do Spark (translate/regexp_replace),
# que rodam inteiramente na JVM, sem esse custo de serialização.

# Mapa de acentos comuns do português — cobre os caracteres esperados
# nos slugs de produto (categoria-marca-tamanho-variante).
_ACENTOS_ORIGEM  = "áàãâäéèêëíìîïóòõôöúùûüçñÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇÑ"
_ACENTOS_DESTINO = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"


def normalizar_codigo_produto_col(coluna):
    """
    Versão 100% nativa (sem UDF Python) da normalização de produto:
    minúsculas, sem acento, sem espaço, sem hífen duplicado/nas pontas.
    """
    return (
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    translate(
                        trim(lower(coluna)),
                        _ACENTOS_ORIGEM, _ACENTOS_DESTINO,
                    ),
                    r"\s+", "",       # remove espaços em qualquer posição
                ),
                r"-{2,}", "-",         # colapsa hífens duplicados
            ),
            r"^-+|-+$", "",           # remove hífen nas pontas
        )
    )

## Leitura da Bronze

In [0]:
df_bronze = read_delta(BRONZE_ITENS_VENDA_CAIXA_PATH, adls_options)
print(f"Registros lidos da Bronze: {df_bronze.count():,}")

## Leitura da tabela de apoio (vendas_caixa para validar FK)

###Checagem de atualidade — Bronze de vendas_caixa

Physical_vendas_caixa é mantida por outro membro da squad, fora deste pipeline. Sem essa checagem, uma Bronze desatualizada gera um bug silencioso e difícil de diagnosticar (foi exatamente o que causou o descompasso de id_transacao na carga 2024–jun/2026: a Bronze de itens já estava atualizada, mas a de vendas_caixa não, fazendo o JOIN de FK falhar para 100% das linhas sem nenhum erro explícito).

Em vez de confiar apenas num intervalo de tempo fixo entre os dois pipelines, este notebook falha explicitamente e cedo se a Bronze de vendas_caixa estiver desatualizada — assim o problema aparece aqui, de forma clara, em vez de se propagar como "0 linhas" lá na Gold sem explicação.



In [0]:
from datetime import datetime, timezone

LIMITE_HORAS_VENDAS_CAIXA = 48  # ajustar conforme rotina combinada com a squad

df_vendas_check = read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
data_mais_recente = df_vendas_check.agg({"bronze_ingested_at": "max"}).collect()[0][0]

if data_mais_recente is None:
    raise RuntimeError(
        "ERRO: Bronze de vendas_caixa vazia ou sem coluna "
        "'bronze_ingested_at'. Verifique se o pipeline do colega já rodou."
    )

if data_mais_recente.tzinfo is None:
    data_mais_recente = data_mais_recente.replace(tzinfo=timezone.utc)

horas_desde_ingestao = (
    datetime.now(timezone.utc) - data_mais_recente
).total_seconds() / 172800

vendas_caixa_desatualizada = horas_desde_ingestao > LIMITE_HORAS_VENDAS_CAIXA

registrar_metrica_dq(
    tabela="physical_vendas_caixa",
    regra="00_atualidade_bronze_dependencia_externa",
    qtd_registros_afetados=(1 if vendas_caixa_desatualizada else 0),
    qtd_registros_total=1,
    adls_options=adls_options,
)

if vendas_caixa_desatualizada:
    raise RuntimeError(
        f"ERRO: Bronze de vendas_caixa desatualizada ha "
        f"{horas_desde_ingestao:.1f}h (limite: {LIMITE_HORAS_VENDAS_CAIXA}h). "
        f"O pipeline do colega pode nao ter rodado ainda ou falhou. "
        f"Verifique antes de prosseguir -- NAO ajuste o limite so para "
        f"passar por este erro sem investigar a causa."
    )

print(f"[OK] Bronze de vendas_caixa atualizada ha {horas_desde_ingestao:.1f}h "
      f"(limite: {LIMITE_HORAS_VENDAS_CAIXA}h).")

In [0]:
# Bug corrigido: JOIN de FK comparava id_transacao cru dos dois lados,
# sem normalizar espaco/caixa -- na carga nova isso zerou 100% dos
# matches (nenhuma linha batia), mesmo com as transacoes existindo de
# verdade. trim(upper(...)) normaliza os dois lados so para fins de
# comparacao, sem alterar o valor original de id_transacao na saida.
df_vendas_ids = (
    read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
    .select(
        trim(upper(col("id_transacao"))).alias("_id_transacao_ref")
    )
    .distinct()
)
print(f"Transações de apoio disponíveis: {df_vendas_ids.count():,}")


## Verificação de pré-condição

In [0]:
verificar_destino_limpo(
    SILVER_ITENS_VENDA_CAIXA_PATH,
    adls_options,
    permitir_existente=(SILVER_WRITE_MODE == "overwrite"),
)

## Quarentena de PK (id_item_venda nulo ou duplicado)

Único passo que exige separação antes do pass principal,
pois registros com PK inválida são removidos do fluxo.

In [0]:
# Performance: Databricks Serverless NÃO suporta .cache()/.persist()
# (PERSIST TABLE is not supported on serverless compute). Sem essa
# opção, não dá pra evitar 100% a recomputação do df_bronze — mas
# ainda reduzimos de 3 chamadas de .count() para 2, calculando
# qtd_pk_valida uma única vez e reaproveitando o valor abaixo, em vez
# de chamar df_pk_valida.count() de novo no segundo print.
total_bronze = df_bronze.count()

df_pk_valida = separar_quarentena_pk(
    df=df_bronze,
    coluna_pk="id_item_venda",
    quarentena_path=SILVER_QUARENTENA_ITENS_VENDA_CAIXA_PATH,
    adls_options=adls_options,
)

qtd_pk_valida = df_pk_valida.count()  # calculado 1x só, reaproveitado abaixo
qtd_quarentenados_pk = total_bronze - qtd_pk_valida
print(f"[R1 PK] {qtd_quarentenados_pk:,} registro(s) em quarentena.")
print(f"        {qtd_pk_valida:,} registro(s) válidos para processamento.")

## Pass único — tipagem + FK + todas as flags de qualidade

Todas as transformações e flags calculadas em um único DataFrame,
sem count() intermediário. Evita múltiplas varreduras de 3,7M linhas
que causavam timeout no Serverless.

In [0]:
classificar_produto_udf = udf(classificar_produto, StringType())

df_completo = (
    # ── Tipagem ─────────────────────────────────────────────────────────
    df_pk_valida
    .withColumn("id_item_venda",
                expr("TRY_CAST(id_item_venda AS BIGINT)"))
    .withColumn("id_transacao",
                col("id_transacao"))                          # UUID — string
    .withColumn("quantidade",
                expr("TRY_CAST(quantidade AS DOUBLE)"))
    .withColumn("preco_unitario_registro",
                expr("TRY_CAST(preco_unitario_registro AS DECIMAL(10,2))"))
    .withColumn("valor_total_item_original",
                expr("TRY_CAST(valor_total_item AS DECIMAL(10,2))"))
    .drop("valor_total_item")

    # ── JOIN FK (id_transacao) ───────────────────────────────────────────
    # Compara versao normalizada (trim+upper) dos dois lados -- id_transacao
    # original (sem alterar) continua na coluna de saida.
    .join(
        df_vendas_ids.withColumn("_fk_existe", lit(True)),
        trim(upper(col("id_transacao"))) == col("_id_transacao_ref"),
        how="left",
    )
    .withColumn("flag_fk_invalido", col("_fk_existe").isNull())
    .drop("_fk_existe", "_id_transacao_ref")

    # ── Código de barras ausente ─────────────────────────────────────────
    .withColumn("flag_codigo_barras_ausente",
                col("codigo_barras_produto").isNull() |
                (trim(col("codigo_barras_produto")) == ""))

    # ── Instrução do cliente: normalização de produto p/ ranking ─────────
    # Agrupa corretamente variações de escrita (acento/espaço/hífen) do
    # mesmo produto, sem depender de um SKU/EAN oficial que não existe
    # hoje na fonte.
    .withColumn("codigo_produto_normalizado",
                normalizar_codigo_produto_col(col("codigo_barras_produto")))

    # ── Classificação perecível/seco (a partir do código normalizado) ────
    # Movida para ANTES da Regra 3, pois a validação de quantidade abaixo
    # depende de saber se o produto é perecível (permite fracionário,
    # ex.: kg/L) ou seco (deve ser quantidade inteira, unidades).
    .withColumn("categoria_produto",
                classificar_produto_udf(col("codigo_produto_normalizado")))

    # ── Regra 3: quantidade > 0; kg/L permite float, un exige inteiro ────
    # Especificação oficial: "para produtos em kg/L, permitir float; para
    # un, inteiro". Não existe coluna de unidade de medida na origem, então
    # a categoria (perecível = vendido por peso/volume, seco = por unidade)
    # é usada como proxy: perecível aceita fracionário, seco exige inteiro.
    .withColumn(
        "flag_quantidade_invalido",
        col("quantidade").isNull()
        | (col("quantidade") <= 0)
        | (
            (col("categoria_produto") == "seco")
            & (col("quantidade") != floor(col("quantidade")))
        )
    )

    # ── Regra 4: preco_unitario_registro > 0 ────────────────────────────
    .withColumn("flag_preco_invalido",
                col("preco_unitario_registro").isNull() |
                (col("preco_unitario_registro") <= 0))

    # ── Regra 5: consistência do valor total ────────────────────────────
    .withColumn("valor_calculado",
                spark_round(
                    col("preco_unitario_registro") * col("quantidade"), 2
                ))
    .withColumn("diferenca_valor",
                spark_abs(
                    col("valor_total_item_original") - col("valor_calculado")
                ))
    .withColumn("flag_valor_inconsistente",
                col("valor_total_item_original").isNull() |
                (col("diferenca_valor") > 0.15))
    .withColumn("valor_item_analitico",
                when(col("flag_valor_inconsistente"), col("valor_calculado"))
                .otherwise(col("valor_total_item_original")))
)

print("[OK] Pass único concluído — calculando métricas de qualidade...")


## Registro de métricas de DQ

Um único count() com agregações condicionais por regra —
em vez de 6 counts() separados que varriam 3,7M linhas cada.

In [0]:
metricas = df_completo.agg(
    count("*").alias("total"),
    spark_sum(col("flag_fk_invalido").cast("long")).alias("fk_invalido"),
    spark_sum(col("flag_quantidade_invalido").cast("long")).alias("qtd_invalido"),
    spark_sum(col("flag_preco_invalido").cast("long")).alias("preco_invalido"),
    spark_sum(col("flag_valor_inconsistente").cast("long")).alias("valor_inconsistente"),
    spark_sum(col("flag_codigo_barras_ausente").cast("long")).alias("cod_barras_ausente"),
    spark_sum(
        (col("categoria_produto") == "NAO_CLASSIFICADO").cast("long")
    ).alias("nao_classificado"),
).collect()[0]

total = metricas["total"]
print(f"Total de registros válidos (pós-PK): {total:,}")

REGRAS_METRICAS = [
    ("02_fk_id_transacao_orfa",              metricas["fk_invalido"]),
    ("03_quantidade_invalida",               metricas["qtd_invalido"]),
    ("04_preco_unitario_invalido",           metricas["preco_invalido"]),
    ("05_valor_total_item_inconsistente",    metricas["valor_inconsistente"]),
    ("06_categoria_produto_nao_classificado",metricas["nao_classificado"]),
]

for regra, qtd in REGRAS_METRICAS:
    registrar_metrica_dq(
        tabela="physical_itens_venda_caixa",
        regra=regra,
        qtd_registros_afetados=int(qtd or 0),
        qtd_registros_total=int(total),
        adls_options=adls_options,
    )
    print(f"  [{regra}] {int(qtd or 0):,} afetados de {total:,}")

# Registrar PK separadamente (total veio de df_bronze antes da quarentena)
registrar_metrica_dq(
    tabela="physical_itens_venda_caixa",
    regra="01_pk_id_item_venda_nula_ou_duplicada",
    qtd_registros_afetados=int(qtd_quarentenados_pk),
    qtd_registros_total=int(total_bronze),
    adls_options=adls_options,
)

## Metadados de auditoria + escrita em Delta

In [0]:
df_silver_final = adicionar_metadados_silver(df_completo)

write_delta(
    df_silver_final,
    SILVER_ITENS_VENDA_CAIXA_PATH,
    adls_options,
    mode=SILVER_WRITE_MODE,
)

print(f"[OK] Silver gravada em '{SILVER_ITENS_VENDA_CAIXA_PATH}'.")


## Validação final

In [0]:
df_saved = read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)
total_salvo = df_saved.count()
print(f"Total gravado na Silver: {total_salvo:,}")
print(f"Colunas: {df_saved.columns}")
